Negative Image Script
=====================
Short Description:
This script removes specific labeled regions (e.g., bubbles) from a set of images using masks, then saves the cleaned images to a new folder.

To Use with a Different Dataset or Output Location, Change These:

1. Input image folder:
   IMG_DIR = r"path/to/input/images.jpg"
   → Change this to the folder containing your input .jpg images.

2. Mask image folder:
   MASK_DIR = r"path/to/pixel/masks.png"
   → Change this to the folder containing your labeled mask images.

3. Output folder for cleaned images:
   OUT_DIR = r"path/to/save/output/images.jpg"
   → Change this to where you want the processed images to be saved.

4. Image file extension (if different):
   IMG_EXT = ".jpg"
   → Change to match the format of your input images (e.g., ".png", ".bmp").

5. Inpainting and dilation settings:
   DILATE_PIXELS    = 3               # Number of pixels to dilate around the mask
   INPAINT_RADIUS   = 3               # Radius for inpainting
   INPAINT_METHOD   = cv2.INPAINT_TELEA  # Use cv2.INPAINT_NS for Navier-Stokes method

What it does:
- Loads each image and its corresponding mask.
- Identifies regions labeled with class value 1 in the mask.
- Optionally dilates and cleans mask edges.
- Uses OpenCV’s inpainting to remove the labeled area from the image.
- Saves the result to the output folder.


In [ ]:
import os, re, cv2, numpy as np

IMG_DIR  = r"C:/BLENDER/BubbleID/Code/Bubblina/Frames"
MASK_DIR = r"C:/BLENDER/BubbleID/Code/Bubblina/PixelLabelData"
OUT_DIR  = r"C:/BLENDER/BubbleID/Code/CNN/Neg_Bubblina"
os.makedirs(OUT_DIR, exist_ok=True)

IMG_EXT        = ".png"
DILATE_PIXELS  = 3
INPAINT_RADIUS = 3
INPAINT_METHOD = cv2.INPAINT_TELEA

pattern = re.compile(r"^Label_\d+_(frame_\d+)\.(png|bmp|tif|tiff|jpg|jpeg)$", re.IGNORECASE)
kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*DILATE_PIXELS+1,)*2)

# 1) group masks by frame id
by_frame = {}
for fn in os.listdir(MASK_DIR):
    m = pattern.match(fn)
    if not m:
        continue
    frame_id = m.group(1)  # "frame_3756"
    by_frame.setdefault(frame_id, []).append(os.path.join(MASK_DIR, fn))

total = 0
done  = 0

# 2) process each frame once
for frame_id, mask_paths in sorted(by_frame.items()):
    img_path = os.path.join(IMG_DIR, frame_id + IMG_EXT)
    if not os.path.isfile(img_path):
        print(f"image missing for {frame_id}; skipping")
        continue

    img = cv2.imread(img_path)
    if img is None:
        print(f"could not read image for {frame_id}")
        continue

    # build a combined binary mask (any value > 0 counts as bubble)
    combined = np.zeros(img.shape[:2], np.uint8)
    for mp in mask_paths:
        m = cv2.imread(mp, cv2.IMREAD_UNCHANGED)
        if m is None:
            print("could not read mask", mp)
            continue
        if m.ndim == 3:
            mg = cv2.cvtColor(m, cv2.COLOR_BGR2GRAY)
        else:
            mg = m
        combined |= (mg > 0).astype(np.uint8) * 255

    if np.count_nonzero(combined) == 0:
        # nothing to inpaint; just copy the frame
        out_path = os.path.join(OUT_DIR, frame_id + IMG_EXT)
        cv2.imwrite(out_path, img)
        done += 1
        total += 1
        continue

    # optional dilation/cleanup
    if DILATE_PIXELS:
        combined = cv2.dilate(combined, kernel, iterations=7)
        combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel, iterations=3)

    clean = cv2.inpaint(img, combined, INPAINT_RADIUS, INPAINT_METHOD)
    out_path = os.path.join(OUT_DIR, frame_id + IMG_EXT)
    cv2.imwrite(out_path, clean)
    done  += 1
    total += 1

print(f"Finished. Wrote {done}/{total} negatives to: {OUT_DIR}")


In [ ]:
import os, re, cv2, numpy as np

# ───────── USER SETTINGS ─────────
REF_FRAME_PATH = r"C:/BLENDER/BubbleID/Code/Bubblina/frame_200.jpg"
MASK_DIR       = r"C:/BLENDER/BubbleID/Code/Bubblina/PixelLabelData"   # Label_*_frame_####.png
OUT_DIR        = r"C:/BLENDER/BubbleID/Code/CNN/Cropped128_Bubblina_not_bubble"
SAVE_CLEAN_REF = True
CLEAN_REF_PATH = r"C:/BLENDER/BubbleID/Code/CNN/cleaned_reference.jpg"

PATCH          = 128          # crop size
ELLIPTICAL_PASTE = True       # True = paste inside ellipse; False = paste whole rectangle
TILE_DONOR       = True       # True = tile donor (keeps texture sharp); False = resize donor to ROI
LUMA_MATCH       = False      # True = shift donor brightness to match a small ring around ROI
RING_FOR_LUMA    = 6          # pixels around ROI for luma matching
PAD              = 40         # reflect padding for safe pastes at borders
MIN_AREA         = 10         # skip tiny mask specks when cropping
# mask name pattern: Label_7_frame_3756.png  → captures "frame_3756"
MASK_PAT = re.compile(r"^Label_\d+_(frame_\d+)\.(png|bmp|tif|tiff|jpg|jpeg)$", re.IGNORECASE)
# ─────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)

def crop_with_reflect(frame, cx, cy, size=PATCH):
    h, w = frame.shape[:2]
    x0 = int(cx - size//2); y0 = int(cy - size//2)
    x1 = x0 + size;         y1 = y0 + size
    pad_left   = max(0, -x0); pad_top    = max(0, -y0)
    pad_right  = max(0, x1 - w); pad_bottom = max(0, y1 - h)
    if pad_left or pad_top or pad_right or pad_bottom:
        frame = cv2.copyMakeBorder(frame, pad_top, pad_bottom, pad_left, pad_right,
                                   borderType=cv2.BORDER_REFLECT_101)
        x0 += pad_left; x1 += pad_left; y0 += pad_top; y1 += pad_top
    return frame[y0:y1, x0:x1]

def tile_to_size(img, w, h):
    H, W = img.shape[:2]
    reps_y = max(1, int(np.ceil(h / H)))
    reps_x = max(1, int(np.ceil(w / W)))
    big = np.tile(img, (reps_y, reps_x, 1))
    return big[:h, :w].copy()

def luma_of_ring(image_pad, center, axes, ring_px):
    # ring around the ellipse on the padded image
    hard = np.zeros(image_pad.shape[:2], np.uint8)
    cv2.ellipse(hard, center, axes, 0, 0, 360, 255, -1)
    ksz = max(1, 2*ring_px+1)
    erode_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksz, ksz))
    inner = cv2.erode(hard, erode_k, iterations=1)
    ring  = cv2.subtract(hard, inner)
    ring_idx = np.where(ring > 0)
    if ring_idx[0].size == 0:
        return float(cv2.cvtColor(image_pad, cv2.COLOR_BGR2GRAY).mean())
    return float(cv2.cvtColor(image_pad, cv2.COLOR_BGR2GRAY)[ring_idx].mean())

def shift_luma(img, target_mean):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
    delta = target_mean - float(g.mean())
    out = np.clip(img.astype(np.float32) + delta, 0, 255).astype(np.uint8)
    return out

# 1) load reference
ref = cv2.imread(REF_FRAME_PATH)
assert ref is not None, f"Could not read reference: {REF_FRAME_PATH}"
ref_clean = ref.copy()
H, W = ref.shape[:2]

# 2) choose donor background
print("Draw ONE donor box (background to copy). ENTER to accept, ESC to skip.")
donor_box = cv2.selectROI("Donor background", ref, fromCenter=False, showCrosshair=True)
cv2.destroyWindow("Donor background")
dx, dy, dw, dh = map(int, donor_box)
donor = ref[dy:dy+dh, dx:dx+dw].copy() if (dw > 0 and dh > 0) else ref.copy()

# 3) select MULTIPLE bubbles (loop selectROI)
print("Draw a bubble box, ENTER to accept. Repeat for more. ESC when DONE.")
rois = []
preview = ref.copy()
while True:
    box = cv2.selectROI("Select bubbles (ESC to finish)", preview, fromCenter=False, showCrosshair=True)
    x, y, w, h = map(int, box)
    if w <= 0 or h <= 0:
        cv2.destroyWindow("Select bubbles (ESC to finish)")
        break
    rois.append((x, y, w, h))
    cv2.rectangle(preview, (x, y), (x+w, y+h), (0,255,0), 2)

print(f"ROIs captured: {len(rois)}")

# 4) HARD-PASTE donor onto each ROI
dst_pad = cv2.copyMakeBorder(ref_clean, PAD, PAD, PAD, PAD, cv2.BORDER_REFLECT_101)
for (x, y, w, h) in rois:
    x0, y0 = x + PAD, y + PAD
    # donor texture for this ROI
    donor_tex = tile_to_size(donor, w, h) if TILE_DONOR else cv2.resize(donor, (w, h), cv2.INTER_LINEAR)

    if LUMA_MATCH:
        center = (x + w//2 + PAD, y + h//2 + PAD)
        axes   = (max(1, w//2), max(1, h//2))
        target_mean = luma_of_ring(dst_pad, center, axes, RING_FOR_LUMA)
        donor_tex = shift_luma(donor_tex, target_mean)

    if ELLIPTICAL_PASTE:
        # paste only inside ellipse
        center = (x + w//2 + PAD, y + h//2 + PAD)
        axes   = (max(1, w//2), max(1, h//2))
        mask = np.zeros(dst_pad.shape[:2], np.uint8)
        cv2.ellipse(mask, center, axes, 0, 0, 360, 255, -1)
        m_roi = mask[y0:y0+h, x0:x0+w][..., None]  # HxWx1
        dst_roi = dst_pad[y0:y0+h, x0:x0+w]
        dst_pad[y0:y0+h, x0:x0+w] = np.where(m_roi > 0, donor_tex, dst_roi)
    else:
        # paste whole rectangle
        dst_pad[y0:y0+h, x0:x0+w] = donor_tex

ref_clean = dst_pad[PAD:-PAD, PAD:-PAD]

if SAVE_CLEAN_REF:
    cv2.imwrite(CLEAN_REF_PATH, ref_clean)
    print("Saved cleaned reference to:", CLEAN_REF_PATH)

# 5) Use ALL mask files only for crop coordinates; crop from ref_clean
def save_crops_from_masks(clean_img):
    saved = 0
    for fn in sorted(os.listdir(MASK_DIR)):
        m = MASK_PAT.match(fn)
        if not m:
            continue
        frame_id = m.group(1)  # e.g., frame_3756
        mp = os.path.join(MASK_DIR, fn)
        mm = cv2.imread(mp, cv2.IMREAD_UNCHANGED)
        if mm is None:
            print("Could not read mask:", mp)
            continue
        if mm.ndim == 3:
            mm = cv2.cvtColor(mm, cv2.COLOR_BGR2GRAY)
        bin_mask = (mm > 0).astype(np.uint8)
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
        bin_mask = cv2.morphologyEx(bin_mask, cv2.MORPH_OPEN,  k, iterations=1)
        bin_mask = cv2.morphologyEx(bin_mask, cv2.MORPH_CLOSE, k, iterations=1)
        contours, _ = cv2.findContours(bin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for idx, cnt in enumerate(contours, start=1):
            if cv2.contourArea(cnt) < MIN_AREA:
                continue
            M = cv2.moments(cnt)
            if M["m00"] == 0:
                continue
            cx = int(M["m10"]/M["m00"]); cy = int(M["m01"]/M["m00"])
            patch = crop_with_reflect(clean_img, cx, cy, PATCH)
            cv2.imwrite(os.path.join(OUT_DIR, f"{frame_id}_{idx}.png"), patch)
            saved += 1
    print(f"Saved {saved} crops from cleaned reference → {OUT_DIR}")

save_crops_from_masks(ref_clean)
cv2.destroyAllWindows()
